# Parsing and preprocessing walkthrough

**Project:** WhatsApp NLP & Conversation Analytics (MSc Data Science)

This notebook walks through the first stage of the pipeline (`src/preprocessor.py`): parsing a WhatsApp export into a
table, adding date/time features, baseline text cleaning, tokenisation, and the data-quality report.

**Data:** the **fictional** chat `data/sample_chat.txt` (12 invented participants; all names, numbers and links are
made up). No real chat is used or stored in this repository. The same code runs on any export in the supported format
(see `docs/supported_formats.md`).

## 1. Raw file inspection

Supported export format (and only this format): `[M/D/YY, H:MM:SS AM/PM] Sender: message`, system lines
`[...] - text` or `[...] text` (no sender), an optional untimed first line, and continuation lines for multi-line
messages. Dates are read as Month/Day/Year.

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('..') / 'src'))

CHAT_PATH = Path('..') / 'data' / 'sample_chat.txt'
text = CHAT_PATH.read_text(encoding='utf-8')
lines = text.split('\n')
print('Total raw lines:', len(lines))
print('First line (no timestamp, WhatsApp notice):', repr(lines[0][:60]) + '...')
print('Invisible LEFT-TO-RIGHT MARKs (U+200E):', text.count('\u200e'))
print('Lines that start a new row:', sum(1 for l in lines if l.lstrip('\u200e').startswith('[')))

Total raw lines: 172
First line (no timestamp, WhatsApp notice): '\u200eMessages and calls are end-to-end encrypted. Only people in'...
Invisible LEFT-TO-RIGHT MARKs (U+200E): 14
Lines that start a new row: 167


## 2. Parsing

`build_dataset` classifies every line as exactly one of `header`, `system` or `user`; lines that do not start with a
timestamp are continuation lines of the previous message. Invisible marks before the bracket are ignored when a row
is detected, and a timestamped line without a sender is a system event.

In [2]:
import pandas as pd
from preprocessor import build_dataset, run_quality_checks

df, parsing_failures = build_dataset(CHAT_PATH)
print('Parsed rows:', len(df), '| columns:', df.shape[1], '| parsing failures:', len(parsing_failures))
print(df['message_type'].value_counts().to_string())

Parsed rows: 168 | columns: 22 | parsing failures: 0
message_type
user      160
system      7
header      1


## 3. Examples by category

In [3]:
examples = {
    'system: group created': df[df['message'].str.contains('created this group', na=False)].index[0],
    'system: member added': df[df['message'].str.contains(' added ', na=False)].index[0],
    'system: security code': df[df['message'].str.contains('security code changed', na=False)].index[0],
    'user: image placeholder': df[df['media_type'] == 'image'].index[0],
    'user: deleted message': df[df['is_deleted']].index[0],
    'user: forwarded': df[df['is_forwarded']].index[0],
    'user: multi-line message': df[df['message'].str.contains('\n', na=False) & (df['message_type'] == 'user')].index[0],
}
table = df.loc[list(examples.values()), ['line_start', 'sender', 'message_type', 'media_type']].copy()
table.insert(0, 'category', list(examples))
table['message'] = df.loc[list(examples.values()), 'message'].str.replace('\n', ' / ').str.slice(0, 48).values
print(table.to_string(index=False))

                category  line_start          sender message_type media_type                                          message
   system: group created           2             NaN       system        NaN                     Alex Demo created this group
    system: member added           3             NaN       system        NaN                      Alex Demo added Isha Demoji
   system: security code           4             NaN       system        NaN Isha Demoji's security code changed. Tap to lear
 user: image placeholder           6    Anya Mockson         user      image                                  <image omitted>
   user: deleted message          20     Isha Demoji         user        NaN                         This message was deleted
         user: forwarded          28   Tara Fakenham         user        NaN [Forwarded] Weekly schedule: practice on Monday 
user: multi-line message         135 Kabir Fictional         user        NaN first line of a note / second line /  / l

## 4. Date/time features

Timestamps are used exactly as written in the export (Month/Day/Year, 12-hour clock). The time zone is unknown and is
never assumed.

In [4]:
cols = ['line_start', 'datetime', 'day_name', 'hour', 'period']
print(df[cols].dropna(subset=['datetime']).head(6).to_string(index=False))
print('First / last timestamp:', df['datetime'].min(), '/', df['datetime'].max())

 line_start            datetime day_name  hour period
          2 2026-03-10 08:30:00  Tuesday   8.0  08-09
          3 2026-03-10 08:30:00  Tuesday   8.0  08-09
          4 2026-03-10 08:30:00  Tuesday   8.0  08-09
          5 2026-03-10 09:02:23  Tuesday   9.0  09-10
          6 2026-03-10 10:27:44  Tuesday  10.0  10-11
          7 2026-03-10 12:31:34  Tuesday  12.0  12-13
First / last timestamp: 2026-03-10 08:30:00 / 2026-03-22 06:52:15


## 5. Baseline text cleaning and tokenisation

Cleaning is deliberately non-aggressive: invisible marks are removed, whitespace is collapsed and text is lower-cased.
Tokenisation splits on whitespace and strips punctuation from the **edges** of tokens: ASCII punctuation, other Unicode
punctuation such as the Devanagari danda `।` and curly quotes, while letters, digits and emoji are kept. Curly
apostrophes are normalised, so `don’t` and `don't` are the same token. Stopword removal uses an English-only list;
no Hindi/Marathi stopword list is used.

In [5]:
from preprocessor import clean_text, tokenize, remove_stopwords

for raw in ['Don\u2019t forget the \u201cplan\u201d tomorrow!', '\u0915\u0943\u092a\u092f\u093e \u0909\u092a\u0938\u094d\u0925\u093f\u0924 \u0930\u0939\u093e\u0964',
            'hi\U0001F600 \U0001F44D done...', '(see you at the ground)']:
    tokens = tokenize(clean_text(raw))
    print(repr(raw))
    print('   tokens          :', tokens)
    print('   without stopwords:', remove_stopwords(tokens))

'Don’t forget the “plan” tomorrow!'
   tokens          : ["don't", 'forget', 'the', 'plan', 'tomorrow']
   without stopwords: ['forget', 'plan', 'tomorrow']
'कृपया उपस्थित रहा।'
   tokens          : ['कृपया', 'उपस्थित', 'रहा']
   without stopwords: ['कृपया', 'उपस्थित', 'रहा']
'hi😀 👍 done...'
   tokens          : ['hi😀', '👍', 'done']
   without stopwords: ['hi😀', '👍', 'done']
'(see you at the ground)'
   tokens          : ['see', 'you', 'at', 'the', 'ground']
   without stopwords: ['see', 'ground']


## 6. Media and deleted-message placeholders

Known WhatsApp placeholders are recognised so that they never become text: bracketed ones such as
`<image omitted>` or `<attached: ...>` anywhere in a message, and bare ones such as `image omitted` or
`You deleted this message` only when they are the whole message (ordinary sentences that mention them stay text).

In [6]:
from preprocessor import classify_media, is_deleted_placeholder

for sample in ['<image omitted>', 'sticker omitted', '<attached: 00000001-PHOTO.jpg>', 'Missed voice call',
               'I said image omitted yesterday', 'You deleted this message', 'the file was deleted by mistake']:
    print(f'{sample!r:42} media={classify_media(sample)!s:11} deleted={is_deleted_placeholder(sample)}')

'<image omitted>'                          media=image       deleted=False
'sticker omitted'                          media=sticker     deleted=False
'<attached: 00000001-PHOTO.jpg>'           media=attachment  deleted=False
'Missed voice call'                        media=call        deleted=False
'I said image omitted yesterday'           media=None        deleted=False
'You deleted this message'                 media=None        deleted=True
'the file was deleted by mistake'          media=None        deleted=False


## 7. Data-quality report

In [7]:
report = run_quality_checks(df, parsing_failures)
for key in ['total_rows', 'duplicate_rows', 'empty_messages', 'invalid_timestamps', 'parsing_failures_count',
            'unexpected_users', 'unusually_long_messages_count']:
    print(f'{key:32}', report[key])
print('Distinct senders:', len(report['unique_senders']))

total_rows                       168
duplicate_rows                   0
empty_messages                   0
invalid_timestamps               1
parsing_failures_count           0
unexpected_users                 []
unusually_long_messages_count    1
Distinct senders: 12


## 8. Limitations and next step

* Only the bracketed `[M/D/YY, H:MM:SS AM/PM]` format is supported; Android "date, time - Sender: text" exports,
  24-hour clocks, timestamps without seconds, 4-digit years and Day/Month dates are not (see `docs/supported_formats.md`).
* Tokenisation is whitespace-based; stopwords are English-only; no language identification is attempted.
* `build_dataset` does not write files. The command-line script `scripts/run_full_pipeline.py` runs this stage and all
  later stages (n-grams, TF-IDF, Script Mix, Key Terms, conversation analytics) and writes the outputs.